# PKG regularity coverage — steady, scheduled, occasional (v1.0)

**Question.** How much of the money moves in relationships that repeat, and which definition catches it? The
periodicity tables call many large, repeating flows "casual" (e.g. a payer sending several invoices a month on changing
days). Before more work relies on them (signal panel, liquidity, the data center app), measure the gap.

**Words** (the same in every PKG notebook and app from now on):

| word | rule | answers |
|---|---|---|
| **scheduled** | the periodicity tables (`pkg_cust_reg_stream`) found a rhythm: daily, weekly, biweekly, monthly, quarterly | when is the next payment due, is it late |
| **steady** | paid in ≥ 8 of the last 12 months and ≥ 4 of the last 6 incl. this one, average of the last 3 months ≤ 1.5 × median of the last 6 (the contagion study's regular payer) | is this a relationship that carries money every month |
| **frequent, uneven** | ≥ 8 of 12 months, not steady (level jumps) | active but unpredictable amounts |
| **occasional** | 3–7 of the last 12 months | comes and goes |
| **rare** | 1–2 of the last 12 months | one-off |

A stream is counted at month m if it paid in m. Scheduled and steady overlap; the table shows every combination.

**Sections.** §1 inputs · §2 classes for every stream-month · §3 coverage tables (dollars and streams by class,
side, direction, size; customer coverage; payments per month) · §4 burst test: would the large steady but unscheduled
streams get a rhythm if payments a few days apart were counted as one? (daily edges, sample) · §5 answer sheet.

One output per cell; tables are pandas; csv files in `OUT_DIR`.

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import os, sys, time, json, math, hashlib, warnings, threading, contextlib
import numpy as np
import pandas as pd

for _a, _t in [("object0", object), ("bool8", np.bool_), ("int0", np.intp), ("uint0", np.uintp), ("bool", np.bool_)]:
    if not hasattr(np, _a):
        setattr(np, _a, _t)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _w in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_w)

from pyspark.sql import SparkSession, functions as F, Window as W

DB = "bdahd01p_dlcdi1_cdi_tm"
CONFIG = dict(
    PAIR_TABLE     = f"{DB}.pkg_cust_pair_month_v4",   # cust, dir, scope (int/ext/self), other, amount, n_txn, o_self, month
    REG_STREAM     = f"{DB}.pkg_cust_reg_stream",      # periodicity: cust, dir, side (ext/int/own), other, month, cadence
    DIM_TABLE      = f"{DB}.pkg_cust_dim",             # mdm_id, customer_name, is_tm, party_type, pwr_shared
    EDGE_DAILY     = f"{DB}.pkg_edge_daily",           # src_id, src_kind, dst_id, dst_kind, txn_date, amount, n_txn, month
    CPTY_MAP       = f"{DB}.pkg_cpty_entity_map_v2",   # cpty_id -> ent_l2 (= pair table `other` for external)
    CPTY_DIM       = f"{DB}.pkg_cpty_dim",             # cpty_id -> cpty_name
    POPULATION     = "tm_orgs",       # "tm_orgs" (is_tm, organisations, power id not shared) or "all"
    ORG_PARTY_TYPES = ["O"],
    LAST_MONTH     = None,            # None = latest month in both the pair and periodicity tables
    ANALYSIS_MONTHS = 6,              # months measured (each needs 12 months of history before it)
    STEADY_MAX     = 1.5,             # average of the last 3 months <= 1.5 x median of the last 6
    SIZE_EDGES     = [10_000, 100_000, 1_000_000, 10_000_000],   # monthly dollars of the stream
    TOP_LIST       = 40,              # rows in the list of largest steady, unscheduled streams
    BURST_SAMPLE   = 3_000,           # §4: largest steady, unscheduled streams tested on daily edges
    BURST_DAYS     = 3,               # payments this many days apart or closer count as one payment
    BURST_MONTHS   = 12,              # months of daily edges read for the test
    CADENCES       = {"weekly": (5, 9, 2), "biweekly": (12, 17, 3),          # biweekly also catches twice a month
                      "monthly": (26, 35, 5), "quarterly": (80, 100, 12)},   # median gap lo..hi days, on-time tolerance
    ONTIME_MIN     = 0.70,            # share of gaps within the tolerance (as in periodicity)
    MIN_EVENTS     = 4,               # payments (after grouping) needed to judge a rhythm
    SHUFFLE_PARTITIONS = 400,
    OUT_DIR        = "../metrics/regularity_coverage",
)

spark = (SparkSession.builder.appName("pkg_regularity_coverage")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "8g").config("spark.executor.memoryOverhead", "3g")
         .config("spark.executor.cores", "2").config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g").config("spark.dynamicAllocation.maxExecutors", "256")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))
         .enableHiveSupport().getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:
    _L = spark._jvm.org.apache.logging.log4j
    for _n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
               "org.apache.spark.sql.catalyst.util.package", "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    pass
spark.conf.set("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))

try:
    from IPython.display import display, HTML
    _NB = "ipykernel" in sys.modules
except Exception:
    _NB = False


def _fmt(v):
    if v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))):
        return ""
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        a = abs(float(v))
        if a >= 1000:
            return f"{v:,.0f}"
        if a >= 10:
            return f"{v:,.1f}"
        return f"{v:,.3f}".rstrip("0").rstrip(".") if a else "0"
    return str(v)


def _html(df):
    d = df.copy()
    for c in d.columns:
        d[c] = d[c].map(_fmt)
    return d.to_html(index=False, border=0, escape=True)


class _Cell:
    """Progress lines and result tables of one cell, rendered as a single output."""
    def start(self, title):
        self.title, self.lines, self.tables = title, [], []
        self.h = display(HTML(""), display_id=True) if _NB else None
        self._draw()

    def log(self, txt, replace_last=False):
        if replace_last and self.lines:
            self.lines[-1] = txt
        else:
            self.lines.append(txt)
        self._draw()
        if not _NB:
            print("   ", txt, flush=True)

    def show(self, *blocks):
        self.tables = list(blocks)
        self._draw()
        if not _NB:
            for lab, df in blocks:
                print(f"\n== {lab}")
                print(df.to_string(index=False, max_rows=80) if len(df) else "(empty)")

    def _draw(self):
        if getattr(self, "h", None) is None:
            return
        css = ("<style>.pkgc table{border-collapse:collapse;font-size:12px;margin:2px 0 12px}"
               ".pkgc th{background:#f1f0ec;color:#0b0b0b;text-align:left;padding:3px 8px;border-bottom:1px solid #b8b7b0}"
               ".pkgc td{padding:2px 8px;border-bottom:1px solid #e6e5e0}"
               ".pkgc h4{margin:10px 0 3px;font-size:13px}.pkgc .pg{color:#8a8985;font-size:12px}</style>")
        body = f"<div class='pkgc'><h3 style='margin:4px 0'>{self.title}</h3>"
        body += "".join(f"<div class='pg'>{x}</div>" for x in self.lines)
        body += "".join(f"<h4>{lab}</h4>{_html(df)}" for lab, df in self.tables)
        self.h.update(HTML(css + body + "</div>"))


CELL = _Cell()


@contextlib.contextmanager
def step(name):
    t0 = time.time()
    CELL.log(f"running: {name} (started {time.strftime('%H:%M')})")
    stop = threading.Event()

    def _tick():
        while not stop.wait(30):
            msg = f"running: {name} ({(time.time() - t0) / 60:.1f} min"
            try:
                st = spark.sparkContext.statusTracker()
                ids = st.getActiveStageIds()
                if ids:
                    info = st.getStageInfo(max(ids))
                    msg += f", stage {info.stageId}: {info.numCompletedTasks:,}/{info.numTasks:,} tasks"
            except Exception:
                pass
            if _NB:
                CELL.log(msg + ")", replace_last=True)

    th = threading.Thread(target=_tick, daemon=True)
    th.start()
    try:
        yield
    finally:
        stop.set()
        mins = (time.time() - t0) / 60
        if _NB:
            CELL.log(f"done: {name} ({mins:.1f} min)", replace_last=True)
        else:
            print(f"    done: {name} ({mins:.1f} min)", flush=True)


ANSWERS = []


def ans(q, a):
    ANSWERS.append(dict(question=q, answer=a))


def table_months(tbl):
    try:
        p = spark.sql(f"SHOW PARTITIONS {tbl}").toPandas().iloc[:, 0]
        m = sorted({x.split("month=")[1].split("/")[0] for x in p if "month=" in x})
        if m:
            return m
    except Exception:
        pass
    return sorted(r[0] for r in spark.table(tbl).select("month").distinct().collect())


def mi_of(m):
    return int(m[:4]) * 12 + int(m[5:7])


def m_of(i):
    return f"{(i - 1) // 12:04d}-{(i - 1) % 12 + 1:02d}"


mi_col = lambda c: (F.substring(F.col(c), 1, 4).cast("int") * 12 + F.substring(F.col(c), 6, 2).cast("int")).cast("int")
pct = lambda x, cols: x.assign(**{c: x[c].map(lambda v: "" if pd.isna(v) else f"{v:.1%}") for c in cols if c in x})
OUT_DIR = CONFIG["OUT_DIR"]
os.makedirs(OUT_DIR, exist_ok=True)
CLASS_ORDER = ["scheduled and steady", "scheduled, not steady", "steady, not scheduled", "frequent, uneven",
               "occasional", "rare"]
print("setup done")

## §1 · Inputs

Months present in both the pair table and the periodicity stream table; the last `ANALYSIS_MONTHS` of them are
measured, each with 12 months of pair history before it.

In [ ]:
CELL.start("§1 · Inputs")
with step("months and population"):
    _pm, _rm = table_months(CONFIG["PAIR_TABLE"]), table_months(CONFIG["REG_STREAM"])
    _both = sorted(set(_pm) & set(_rm))
    if CONFIG["LAST_MONTH"]:
        _both = [m for m in _both if m <= CONFIG["LAST_MONTH"]]
    if not _both:
        raise ValueError("No month in both the pair table and the periodicity stream table.")
    LAST = _both[-1]
    AM = [m for m in _both if m in _pm][-CONFIG["ANALYSIS_MONTHS"]:]
    HIST0 = m_of(mi_of(AM[0]) - 11)
    _miss_hist = [m for m in (m_of(i) for i in range(mi_of(HIST0), mi_of(LAST) + 1)) if m not in set(_pm)]
    dim = spark.table(CONFIG["DIM_TABLE"])
    _dc = set(dim.columns)
    _pop = dim.select(F.col("mdm_id").cast("string").alias("cust"),
                      *[F.col(c) for c in ["customer_name", "is_tm", "party_type", "pwr_shared"] if c in _dc])
    if CONFIG["POPULATION"] == "tm_orgs":
        cond = F.lit(True)
        if "is_tm" in _dc:
            cond = cond & (F.col("is_tm").cast("boolean") == F.lit(True))
        if "party_type" in _dc:
            cond = cond & F.col("party_type").isin(CONFIG["ORG_PARTY_TYPES"])
        if "pwr_shared" in _dc:
            cond = cond & (F.coalesce(F.col("pwr_shared").cast("boolean"), F.lit(False)) == F.lit(False))
        _pop = _pop.where(cond)
    POP = _pop.select("cust", "customer_name").dropDuplicates(["cust"]).cache()
    n_pop = POP.count()
inputs = pd.DataFrame([dict(item="months measured", value=f"{AM[0]} .. {AM[-1]} ({len(AM)})"),
                       dict(item="pair history read from", value=HIST0),
                       dict(item="pair months missing in that range", value=", ".join(_miss_hist) or "none"),
                       dict(item="periodicity months available", value=f"{_rm[0]} .. {_rm[-1]} ({len(_rm)})"),
                       dict(item="population", value=f"{CONFIG['POPULATION']}: {n_pop:,} customers")])
CELL.show(("Inputs", inputs))

## §2 · A class for every stream-month

Stream = customer × direction × side (`ext` outside PNC, `int` another PNC customer) × counterparty, from the pair
table (own accounts at other banks and self transfers left out: they are counted apart). Scheduled = the stream has a
row with a cadence in the periodicity stream table that month.

In [ ]:
CELL.start("§2 · Classes")
with step("stream windows over the pair table"):
    _pc = set(spark.table(CONFIG["PAIR_TABLE"]).columns)
    _ntx = F.col("n_txn").cast("double") if "n_txn" in _pc else F.lit(None).cast("double")
    _own = (F.coalesce(F.col("o_self").cast("int"), F.lit(0)) == 1) if "o_self" in _pc else F.lit(False)
    pair = (spark.table(CONFIG["PAIR_TABLE"])
            .where(F.col("month").between(HIST0, AM[-1]) & F.col("scope").isin("int", "ext", "self"))
            .withColumn("side", F.when(F.col("scope") == "self", F.lit("own")).when(_own, F.lit("own")).otherwise(F.col("scope")))
            .select(F.col("cust").cast("string").alias("cust"), "dir", "side", F.col("other").cast("string").alias("other"),
                    mi_col("month").alias("mi"), F.col("amount").cast("double").alias("amount"), _ntx.alias("n_txn"))
            .join(POP.select("cust"), "cust", "left_semi")
            .groupBy("cust", "dir", "side", "other", "mi").agg(F.sum("amount").alias("amount"), F.sum("n_txn").alias("n_txn"))
            .where(F.col("amount") > 0))
    _w = W.partitionBy("cust", "dir", "side", "other").orderBy("mi")
    win = (pair
           .withColumn("cnt6", F.count(F.lit(1)).over(_w.rangeBetween(-5, 0)))
           .withColumn("cnt12", F.count(F.lit(1)).over(_w.rangeBetween(-11, 0)))
           .withColumn("amt12", F.sum("amount").over(_w.rangeBetween(-11, 0)))
           .withColumn("ntx12", F.sum("n_txn").over(_w.rangeBetween(-11, 0)))
           .withColumn("sd12", F.stddev("amount").over(_w.rangeBetween(-11, 0)))
           .withColumn("last6", F.collect_list("amount").over(_w.rangeBetween(-5, 0)))
           .withColumn("last3", F.collect_list("amount").over(_w.rowsBetween(-2, 0)))
           .where(F.col("mi").between(mi_of(AM[0]), mi_of(AM[-1])))
           .withColumn("med6", F.expr("element_at(array_sort(last6), cast(ceil(size(last6) / 2.0) as int))"))
           .withColumn("avg3", F.expr("aggregate(last3, cast(0 as double), (a, x) -> a + x) / size(last3)"))
           .withColumn("steady", (F.col("cnt6") >= 4) & (F.col("cnt12") >= 8) & (F.col("avg3") <= CONFIG["STEADY_MAX"] * F.col("med6")))
           .drop("last6", "last3"))

with step("scheduled: rows with a cadence in the periodicity stream table"):
    _rc = set(spark.table(CONFIG["REG_STREAM"]).columns)
    _cad = F.col("cadence").cast("string") if "cadence" in _rc else F.lit(None).cast("string")
    reg = (spark.table(CONFIG["REG_STREAM"]).where(F.col("month").isin(AM))
           .select(F.col("cust").cast("string").alias("cust"), "dir", F.col("side").cast("string").alias("side"),
                   F.col("other").cast("string").alias("other"), mi_col("month").alias("mi"), _cad.alias("cadence"))
           .where(F.col("cadence").isNotNull())
           .groupBy("cust", "dir", "side", "other", "mi").agg(F.max("cadence").alias("cadence")))
    sm = (win.join(reg, ["cust", "dir", "side", "other", "mi"], "left")
          .withColumn("scheduled", F.col("cadence").isNotNull())
          .withColumn("cls", F.when(F.col("scheduled") & F.col("steady"), "scheduled and steady")
                      .when(F.col("scheduled"), "scheduled, not steady")
                      .when(F.col("steady"), "steady, not scheduled")
                      .when(F.col("cnt12") >= 8, "frequent, uneven")
                      .when(F.col("cnt12") >= 3, "occasional").otherwise("rare"))
          .withColumn("size_band", F.when(F.col("amount") < CONFIG["SIZE_EDGES"][0], "under 10K")
                      .when(F.col("amount") < CONFIG["SIZE_EDGES"][1], "10K-100K")
                      .when(F.col("amount") < CONFIG["SIZE_EDGES"][2], "100K-1M")
                      .when(F.col("amount") < CONFIG["SIZE_EDGES"][3], "1M-10M").otherwise("10M+"))
          .withColumn("ppm", F.col("ntx12") / F.col("cnt12"))     # payments per active month, last 12
          .withColumn("ppm_band", F.when(F.col("ppm").isNull(), "unknown").when(F.col("ppm") < 1.5, "1")
                      .when(F.col("ppm") < 3.5, "2-3").when(F.col("ppm") < 8.5, "4-8").when(F.col("ppm") < 20.5, "9-20")
                      .otherwise("21+"))
          .persist())
    n_sm = sm.count()
    # periodicity streams that the pair table does not carry this month (should be few)
    _reg_only = reg.join(win.select("cust", "dir", "side", "other", "mi"), ["cust", "dir", "side", "other", "mi"], "left_anti").count()
CELL.show(("Stream-months measured", pd.DataFrame([dict(stream_months=n_sm, scheduled_rows_not_in_pair_table=_reg_only)])))

## §3 · Coverage

1. Dollars and streams by class, for each direction and side (average month).
2. The same by monthly size of the stream (incoming and outgoing, PNC and outside together).
3. Payments per active month by class: steady but unscheduled streams that are paid many times a month look like
   invoice batches, which a rhythm test on single payment days misses.
4. Cadence of the scheduled streams, by whether they are steady.
5. Customer coverage: share of each customer's dollars that is scheduled, and scheduled or steady (quantiles over
   customers with any money that way).
6. The largest steady, unscheduled streams (names; stays in the notebook and csv).

In [ ]:
CELL.start("§3 · Coverage")
nm = len(AM)
with step("dollars and streams by class"):
    t1 = (sm.groupBy("dir", "side", "cls").agg((F.sum("amount") / nm).alias("usd_per_month"),
                                                 (F.count(F.lit(1)) / nm).alias("streams_per_month")).toPandas())
    t1["usd_share"] = t1.usd_per_month / t1.groupby(["dir", "side"]).usd_per_month.transform("sum")
    t1["stream_share"] = t1.streams_per_month / t1.groupby(["dir", "side"]).streams_per_month.transform("sum")
    t1 = t1.assign(_o=t1.cls.map({c: i for i, c in enumerate(CLASS_ORDER)})).sort_values(["dir", "side", "_o"]).drop(columns="_o")
    t1w = t1.pivot_table(index="cls", columns=["dir", "side"], values="usd_share", aggfunc="first").reindex(CLASS_ORDER)
    t1w.columns = [f"{d} {s}" for d, s in t1w.columns]
    t1w = t1w.reset_index().rename(columns={"cls": "class"})
with step("by size of the stream"):
    t2 = (sm.where(F.col("side") != "own").groupBy("size_band", "cls").agg(F.sum("amount").alias("usd"), F.count(F.lit(1)).alias("n"))
          .toPandas())
    t2w = t2.pivot_table(index="size_band", columns="cls", values="usd", aggfunc="sum").fillna(0)
    t2w = t2w.div(t2w.sum(axis=1), axis=0).reindex(["under 10K", "10K-100K", "100K-1M", "1M-10M", "10M+"])
    t2w = t2w[[c for c in CLASS_ORDER if c in t2w.columns]].reset_index()
    t2s = t2.groupby("size_band").agg(usd=("usd", "sum"), streams=("n", "sum"))
    t2w["usd_per_month"] = t2w.size_band.map(t2s.usd / nm)
    t2w["streams_per_month"] = t2w.size_band.map(t2s.streams / nm)
with step("payments per active month by class"):
    t3 = (sm.where(F.col("side") != "own").groupBy("cls", "ppm_band").agg(F.sum("amount").alias("usd")).toPandas())
    t3w = t3.pivot_table(index="cls", columns="ppm_band", values="usd", aggfunc="sum").fillna(0)
    t3w = t3w.div(t3w.sum(axis=1), axis=0).reindex(CLASS_ORDER)
    t3w = t3w[[c for c in ["1", "2-3", "4-8", "9-20", "21+", "unknown"] if c in t3w.columns]].reset_index().rename(columns={"cls": "class"})
with step("cadence of scheduled streams"):
    t4 = (sm.where(F.col("scheduled")).groupBy("cadence", "steady").agg(F.sum("amount").alias("usd"), F.count(F.lit(1)).alias("n"))
          .toPandas())
    t4["steady"] = np.where(t4.steady, "steady", "not steady")
    t4w = t4.pivot_table(index="cadence", columns="steady", values="usd", aggfunc="sum").fillna(0)
    t4w["usd_per_month"] = t4w.sum(axis=1) / nm
    t4w["steady_share"] = t4w.get("steady", 0) / (t4w.usd_per_month * nm)
    t4w = t4w.reset_index()[["cadence", "usd_per_month", "steady_share"]].sort_values("usd_per_month", ascending=False)
with step("customer coverage"):
    _cc = (sm.where(F.col("side") != "own").groupBy("cust", "dir", "mi")
           .agg(F.sum("amount").alias("a"),
                F.sum(F.when(F.col("scheduled"), F.col("amount")).otherwise(0.0)).alias("s"),
                F.sum(F.when(F.col("scheduled") | F.col("steady"), F.col("amount")).otherwise(0.0)).alias("ss"))
           .groupBy("cust", "dir").agg((F.sum("s") / F.sum("a")).alias("scheduled_share"),
                                       (F.sum("ss") / F.sum("a")).alias("scheduled_or_steady_share")))
    qs = [0.25, 0.5, 0.75, 0.9]
    t5 = []
    for d in ["in", "out"]:
        x = _cc.where(F.col("dir") == d)
        n = x.count()
        for c in ["scheduled_share", "scheduled_or_steady_share"]:
            q = x.approxQuantile(c, qs, 0.001)
            t5.append(dict(dir=d, measure=c, customers=n, share_with_none=x.where(F.col(c) <= 0).count() / max(n, 1),
                           **{f"p{int(100 * k)}": v for k, v in zip(qs, q)}))
    t5 = pd.DataFrame(t5)
with step("largest steady, unscheduled streams"):
    _top = (sm.where((F.col("cls") == "steady, not scheduled") & (F.col("side") != "own") & (F.col("mi") == mi_of(AM[-1])))
            .orderBy(F.desc("amt12")).limit(max(CONFIG["TOP_LIST"], CONFIG["BURST_SAMPLE"]))
            .select("cust", "dir", "side", "other", "amt12", "cnt12", "ppm", (F.col("sd12") / (F.col("amt12") / F.col("cnt12"))).alias("cv12"))
            .toPandas())
    _nm = POP.join(spark.createDataFrame(_top[["cust"]].drop_duplicates()), "cust").toPandas().set_index("cust").customer_name
    _int = _top.other[_top.side == "int"].unique().tolist()
    _onm = (dim.select(F.col("mdm_id").cast("string").alias("other"), "customer_name").where(F.col("other").isin(_int)).toPandas()
            .drop_duplicates("other").set_index("other").customer_name) if _int else pd.Series(dtype=object)
    _ext = _top.other[_top.side == "ext"].unique().tolist()
    try:
        _enm = (spark.table(CONFIG["CPTY_MAP"]).select(F.col("cpty_id").cast("string").alias("cpty_id"), F.col("ent_l2").cast("string").alias("other"))
                .join(spark.createDataFrame(pd.DataFrame({"other": _ext})), "other")
                .join(spark.table(CONFIG["CPTY_DIM"]).select(F.col("cpty_id").cast("string").alias("cpty_id"), "cpty_name"), "cpty_id", "left")
                .groupBy("other").agg(F.max("cpty_name").alias("nm")).toPandas().set_index("other").nm) if _ext else pd.Series(dtype=object)
    except Exception:
        _enm = pd.Series(dtype=object)
    _top["customer"] = _top.cust.map(_nm)
    _top["counterparty"] = np.where(_top.side == "int", _top.other.map(_onm), _top.other.map(_enm))
    _top["counterparty"] = _top.counterparty.fillna("(no name)")
    TOP = _top
    t6 = _top.head(CONFIG["TOP_LIST"])[["customer", "dir", "side", "counterparty", "amt12", "cnt12", "ppm", "cv12"]].rename(
        columns={"amt12": "usd_12_months", "cnt12": "months_paid", "ppm": "payments_per_month", "cv12": "monthly_cv"})
for _n, _d in [("cov_by_class", t1), ("cov_by_size", t2w), ("cov_payments_per_month", t3w), ("cov_cadence", t4w),
               ("cov_customers", t5), ("steady_unscheduled_top", _top)]:
    _d.to_csv(os.path.join(OUT_DIR, f"{_n}.csv"), index=False)
CELL.show(("Share of dollars by class, per direction and side (average month)", pct(t1w, [c for c in t1w.columns if c != "class"])),
          ("Dollars and streams per month by class", pct(t1.rename(columns={"cls": "class"}), ["usd_share", "stream_share"])),
          ("Share of dollars by class, by monthly size of the stream (own accounts left out)", pct(t2w, CLASS_ORDER)),
          ("Payments per active month: share of each class's dollars", pct(t3w, ["1", "2-3", "4-8", "9-20", "21+", "unknown"])),
          ("Scheduled streams: dollars per month by cadence, and the share that is also steady", pct(t4w, ["steady_share"])),
          ("Customers: share of their dollars that is scheduled / scheduled or steady", pct(t5, ["share_with_none", "p25", "p50", "p75", "p90"])),
          (f"Largest steady, unscheduled streams in {AM[-1]} (12-month dollars)", t6))
for d in ["in", "out"]:
    x = t1[(t1.dir == d) & (t1.side != "own")].groupby("cls").usd_per_month.sum()
    tot = x.sum()
    if tot > 0:
        ans(f"{d}: share of dollars scheduled / steady but not scheduled / neither",
            f"{(x.get('scheduled and steady', 0) + x.get('scheduled, not steady', 0)) / tot:.0%} / "
            f"{x.get('steady, not scheduled', 0) / tot:.0%} / "
            f"{(tot - x.get('scheduled and steady', 0) - x.get('scheduled, not steady', 0) - x.get('steady, not scheduled', 0)) / tot:.0%}")

## §4 · Burst test: would grouping payments give the large steady streams a rhythm?

For the `BURST_SAMPLE` largest steady but unscheduled streams (last measured month), payment days over
`BURST_MONTHS` months from the daily edges. Payment days `BURST_DAYS` apart or closer become one payment (the first
day). The gaps between payments are then tested like periodicity: the median gap falls in a cadence's range and at
least 70% of gaps sit within its tolerance, with at least 4 payments. The same test on raw payment days shows what
grouping adds. Read the result as the size of the prize for a "burst" rule in periodicity v1.2+.

In [ ]:
CELL.start("§4 · Burst test")
burst, burst_sum = pd.DataFrame(), pd.DataFrame()
try:
    with step("payment days of the sample from the daily edges"):
        S = TOP.head(CONFIG["BURST_SAMPLE"])[["cust", "dir", "side", "other", "amt12"]].copy()
        S["sid"] = np.arange(len(S))
        Sd = spark.createDataFrame(S)
        _m0 = m_of(mi_of(AM[-1]) - CONFIG["BURST_MONTHS"] + 1)
        ed = spark.table(CONFIG["EDGE_DAILY"]).where(F.col("month").between(_m0, AM[-1]))
        _emc = set(spark.table(CONFIG["CPTY_MAP"]).columns)
        cmap = (spark.table(CONFIG["CPTY_MAP"]).select(F.col("cpty_id").cast("string").alias("cpty_id"), F.col("ent_l2").cast("string").alias("ent"))
                .groupBy("cpty_id").agg(F.min("ent").alias("ent")))
        _ec = (ed.select(F.col("src_id").cast("string").alias("src_id"), F.upper(F.col("src_kind").cast("string")).alias("src_kind"),
                         F.col("dst_id").cast("string").alias("dst_id"), F.upper(F.col("dst_kind").cast("string")).alias("dst_kind"),
                         F.to_date(F.col("txn_date")).alias("d"), F.col("amount").cast("double").alias("amount")))
        # incoming: customer is dst; outgoing: customer is src
        _in = (_ec.where(F.col("dst_kind").startswith("CUST")).select(F.col("dst_id").alias("cust"), F.lit("in").alias("dir"),
                                                                    F.col("src_id").alias("oid"), F.col("src_kind").alias("okind"), "d", "amount"))
        _out = (_ec.where(F.col("src_kind").startswith("CUST")).select(F.col("src_id").alias("cust"), F.lit("out").alias("dir"),
                                                                     F.col("dst_id").alias("oid"), F.col("dst_kind").alias("okind"), "d", "amount"))
        _e = _in.unionByName(_out).join(F.broadcast(Sd.select("cust").distinct()), "cust", "left_semi")
        _e = (_e.join(cmap.withColumnRenamed("cpty_id", "oid"), "oid", "left")
              .withColumn("side", F.when(F.col("okind").startswith("CUST"), F.lit("int")).otherwise(F.lit("ext")))
              .withColumn("other", F.when(F.col("side") == "int", F.col("oid")).otherwise(F.col("ent"))))
        days = (_e.join(F.broadcast(Sd), ["cust", "dir", "side", "other"], "inner")
                .groupBy("sid", "d").agg(F.sum("amount").alias("amount")).toPandas())

    def _rhythm(dd, group):
        dd = np.sort(np.unique(dd.astype("datetime64[D]").astype(int)))
        if group and len(dd):
            keep, last = [dd[0]], dd[0]
            for x in dd[1:]:
                if x - last > CONFIG["BURST_DAYS"]:
                    keep.append(x)
                last = x
            dd = np.array(keep)
        if len(dd) < CONFIG["MIN_EVENTS"]:
            return "too few payments", len(dd), np.nan, np.nan
        g = np.diff(dd)
        mg = float(np.median(g))
        for cad, (lo, hi, tol) in CONFIG["CADENCES"].items():
            if lo <= mg <= hi:
                ot = float(np.mean(np.abs(g - mg) <= tol))
                return (cad if ot >= CONFIG["ONTIME_MIN"] else f"{cad}-like, uneven"), len(dd), mg, ot
        if mg < 5:
            return "several a week (daily-like)", len(dd), mg, np.nan
        return "no rhythm", len(dd), mg, np.nan

    with step("rhythm test with and without grouping"):
        rows = []
        for sid, g in days.groupby("sid"):
            r0 = _rhythm(g.d.values, False)
            r1 = _rhythm(g.d.values, True)
            rows.append(dict(sid=sid, raw=r0[0], grouped=r1[0], payments_raw=r0[1], payments_grouped=r1[1],
                             median_gap_grouped=r1[2], ontime_grouped=r1[3]))
        burst = S.merge(pd.DataFrame(rows), on="sid", how="left")
        burst["raw"] = burst.raw.fillna("no daily rows found")
        burst["grouped"] = burst.grouped.fillna("no daily rows found")
        _ok = lambda s_: s_.isin(list(CONFIG["CADENCES"]))
        burst_sum = pd.DataFrame([
            dict(test="raw payment days", streams=len(burst), with_rhythm=_ok(burst.raw).mean(),
                 usd_share_with_rhythm=burst.amt12[_ok(burst.raw)].sum() / burst.amt12.sum()),
            dict(test=f"payments {CONFIG['BURST_DAYS']} days apart or closer grouped", streams=len(burst),
                 with_rhythm=_ok(burst.grouped).mean(), usd_share_with_rhythm=burst.amt12[_ok(burst.grouped)].sum() / burst.amt12.sum())])
        bt = (burst.groupby("grouped").agg(streams=("sid", "count"), usd_12m=("amt12", "sum")).reset_index()
              .rename(columns={"grouped": "result after grouping"}))
        bt["usd_share"] = bt.usd_12m / bt.usd_12m.sum()
        bt = bt.sort_values("usd_12m", ascending=False)
        burst.to_csv(os.path.join(OUT_DIR, "burst_test.csv"), index=False)
    CELL.show(("Rhythm found, raw days against grouped payments (sample of the largest steady, unscheduled streams)",
               pct(burst_sum, ["with_rhythm", "usd_share_with_rhythm"])),
              ("Result after grouping", pct(bt, ["usd_share"])))
    if len(burst_sum):
        ans("Largest steady, unscheduled streams: share of dollars with a rhythm, raw days / grouped",
            f"{burst_sum.usd_share_with_rhythm.iloc[0]:.0%} / {burst_sum.usd_share_with_rhythm.iloc[1]:.0%}")
except Exception as ex_:
    CELL.show(("Burst test", pd.DataFrame([dict(status=f"skipped: {str(ex_)[:300]}")])))

## §5 · Answer sheet

In [ ]:
CELL.start("§5 · Answer sheet")
A = pd.DataFrame(ANSWERS)
A.to_csv(os.path.join(OUT_DIR, "answers.csv"), index=False)
CELL.show(("Answers", A))